<a href="https://colab.research.google.com/github/iagoxyz01/Prototipo/blob/main/Projeto_Big_Data_Gastos_Eleitorais_TSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Big Data — Gastos eleitorais (TSE)

## Objetivo
Investigar a relação entre despesas de campanha e eleição.

## Escopo
- Presidente — Brasil, 2018, 2022 e 2026.
- Governador — estados do Nordeste (AL, BA, CE, MA, PB, PE, PI, RN e SE).

## Tabelas
1. Candidatos, cargo, estado, partido, gastos contratados e eleito.
2. Top 10 gastos por ano/cargo.
3. Categoria de despesa dos Top 10.

**Métrica de gastos:** total de despesas contratadas (`VR_DESPESA_CONTRATADA`) somadas por `SQ_CANDIDATO`. O notebook utiliza despesas contratadas, pois esse arquivo permite vincular cada lançamento diretamente ao candidato; não usa `SQ_PRESTADOR_CONTAS` como se fosse ID do candidato.

Fonte: Portal de Dados Abertos do TSE. 2026 representa prestação parcial/atualizada e não deve ser comparada como final.

## 1. Instalar/importar bibliotecas

In [ ]:
import io
import re
import zipfile
import unicodedata
from pathlib import Path

import pandas as pd
import requests

ANOS = [2018, 2022, 2026]
ESTADOS_NE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]

DATAS = {
    2018: "07/10/2018",
    2022: "02/10/2022",
    2026: "04/10/2026"
}

API_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"

print("Bibliotecas e configurações carregadas.")

## 2. Funções para consultar e localizar arquivos no TSE

In [ ]:
def consultar_conjunto(slug):
    try:
        resposta = requests.get(API_TSE, params={"id": slug}, timeout=45)
        resposta.raise_for_status()
        dados = resposta.json()
        if dados.get("success"):
            return dados["result"].get("resources", [])
        return None
    except Exception as exc:
        print(f"Catalogo {slug}: {type(exc).__name__}: {exc}")
        return None


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))


def escolher_recurso(recursos, palavras):
    if not recursos:
        return None
    for recurso in recursos:
        nome = normalizar(recurso.get("name", ""))
        if all(p in nome for p in palavras):
            return recurso
    return None


def localizar_recursos(ano):
    candidatos = consultar_conjunto(f"candidatos-{ano}")
    resultados = consultar_conjunto(f"resultados-{ano}")

    slugs_contas = {
        2018: ["prestacao-de-contas-eleitorais-2018"],
        2022: ["dadosabertos-tse-jus-br-dataset-prestacao-de-contas-eleitorais-2022"],
        2026: ["prestacao-de-contas-eleitorais-2026"],
    }
    contas = None
    for slug in slugs_contas.get(ano, []):
        contas = consultar_conjunto(slug)
        if contas is not None:
            break

    if resultados is None:
        resultados = consultar_conjunto(f"resultados-eleitorais-{ano}")

    if candidatos is None or resultados is None or contas is None:
        raise ValueError(
            f"Pacotes TSE ausentes para {ano}: candidaturas={candidatos is not None}, "
            f"resultados={resultados is not None}, contas={contas is not None}"
        )

    recurso_candidatos = (
        escolher_recurso(candidatos, ["consulta", "cand", "brasil"])
        or escolher_recurso(candidatos, ["consulta", "cand"])
        or escolher_recurso(candidatos, ["cand"])
    )
    recurso_despesas = (
        escolher_recurso(contas, ["prestacao", "candidatos"])
        or escolher_recurso(contas, ["candidatos"])
    )
    recurso_resultados = (
        escolher_recurso(resultados, ["votacao", "candidato", "brasil"])
        or escolher_recurso(resultados, ["votacao", "candidato"])
        or escolher_recurso(resultados, ["votacao", "nominal"])
    )
    recursos = {"candidatos": recurso_candidatos, "despesas": recurso_despesas, "resultados": recurso_resultados}
    faltantes = [k for k, v in recursos.items() if v is None]
    if faltantes:
        raise ValueError(
            f"Recursos não encontrados em {ano}: {faltantes}. "
            f"Candidatos={ [r.get('name') for r in candidatos] }; "
            f"Contas={ [r.get('name') for r in contas] }; "
            f"Resultados={ [r.get('name') for r in resultados] }"
        )
    return recursos


## 3. Baixar e ler os arquivos

In [ ]:
import time


def baixar_conteudo(recurso, max_tentativas=3, intervalo=5):
    print("Baixando:", recurso.get("name"))
    erro_final = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            response = requests.get(recurso["url"], timeout=300, stream=True)
            response.raise_for_status()
            buffer = io.BytesIO()
            for bloco in response.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    buffer.write(bloco)
            return buffer.getvalue()
        except Exception as exc:
            erro_final = exc
            print(f"Tentativa {tentativa}/{max_tentativas}: {exc}")
            if tentativa < max_tentativas:
                time.sleep(intervalo)
    raise ConnectionError(f"Download falhou: {erro_final}")


def escolher_nome_csv_zip(arquivo_zip, palavras=(), tipo=None):
    nomes = [n for n in arquivo_zip.namelist() if n.lower().endswith((".csv", ".txt"))]
    if not nomes:
        raise ValueError("ZIP não contém CSV/TXT")
    norm = {n: normalizar(n) for n in nomes}
    if tipo == "candidatos":
        candidatos = [n for n in nomes if "consulta" in norm[n] and "cand" in norm[n]]
        if not candidatos:
            candidatos = [n for n in nomes if "cand" in norm[n] and "despesa" not in norm[n]]
    elif tipo == "despesas_contratadas":
        candidatos = [n for n in nomes if "despesas contratadas candidatos" in norm[n] or ("despesas contratadas" in norm[n] and "cand" in norm[n])]
        if not candidatos:
            candidatos = [n for n in nomes if "despesas contratadas" in norm[n]]
    elif tipo == "despesas":
        candidatos = [n for n in nomes if "despesas pagas candidatos" in norm[n] or "despesas pagas" in norm[n]]
    elif tipo == "resultados":
        candidatos = [n for n in nomes if "votacao candidato" in norm[n] or "votacao nominal" in norm[n]]
    else:
        candidatos = [n for n in nomes if all(p in norm[n] for p in palavras)] or nomes
    candidatos = candidatos or nomes
    br = [n for n in candidatos if "brasil" in norm[n] or norm[n].endswith("_br.csv") or "_br_" in norm[n]]
    candidatos = br or candidatos
    candidatos.sort(key=lambda n: (0 if "brasil" in norm[n] else 1, len(n)))
    return candidatos[0]


def ler_tabela(conteudo, nome_arquivo=None, nrows=None):
    ultimo_erro = None
    for enc in ("latin1", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(io.BytesIO(conteudo), sep=None, engine="python",
                               encoding=enc, nrows=nrows)
        except Exception as exc:
            ultimo_erro = exc
    raise ValueError(f"CSV não foi lido ({nome_arquivo}): {ultimo_erro}")


def baixar_e_ler(recurso, tipo=None):
    conteudo = baixar_conteudo(recurso)
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_nome_csv_zip(z, tipo=tipo)
            print(f"CSV escolhido ({tipo or 'auto'}):", nome)
            with z.open(nome) as stream:
                return ler_tabela(stream.read(), nome)
    return ler_tabela(conteudo, recurso.get("name"))


## 4. Funções para identificar colunas e somar despesas

In [ ]:
def encontrar_coluna(df, alternativas):
    for nome in alternativas:
        if nome in df.columns:
            return nome
    raise ValueError(f"Colunas {alternativas} não encontradas. Colunas: {list(df.columns)}")


# Alias de compatibilidade: aceita também o nome usado em células antigas.
def encontrar_col(df, alternativas):
    return encontrar_coluna(df, alternativas)


def converter_valores_monetarios(serie):
    texto = serie.astype(str).str.strip().str.replace("R$", "", regex=False).str.strip()
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce").fillna(0)


def processar_despesas_candidatos(recurso, candidatos_ids, ano):
    """Usa despesas CONTRATADAS, que contêm SQ_CANDIDATO, evitando vínculo incorreto por prestador."""
    conteudo = baixar_conteudo(recurso)
    ids = set(map(str, candidatos_ids))
    totais = {}
    categorias = []
    nome_csv = None

    def ler_chunks(stream):
        for chunk in pd.read_csv(stream, sep=";", encoding="latin1", chunksize=250_000, low_memory=False):
            id_col = encontrar_coluna(chunk, ["SQ_CANDIDATO"])
            valor_col = encontrar_coluna(chunk, ["VR_DESPESA_CONTRATADA"])
            categoria_col = encontrar_coluna(chunk, ["DS_DESPESA", "DS_ORIGEM_DESPESA"])
            chunk[id_col] = chunk[id_col].astype(str).str.strip()
            chunk = chunk[chunk[id_col].isin(ids)].copy()
            if chunk.empty:
                continue
            chunk[valor_col] = converter_valores_monetarios(chunk[valor_col])
            chunk[categoria_col] = chunk[categoria_col].fillna("Categoria não informada").astype(str)
            soma = chunk.groupby(id_col)[valor_col].sum()
            for cid, valor in soma.items():
                totais[str(cid)] = totais.get(str(cid), 0.0) + float(valor)
            categoria = chunk.groupby([id_col, categoria_col])[valor_col].sum().reset_index()
            categoria.columns = ["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)"]
            categoria["Ano"] = ano
            categorias.append(categoria)

    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome_csv = escolher_nome_csv_zip(z, tipo="despesas_contratadas")
            print("Arquivo de despesas contratado selecionado:", nome_csv)
            with z.open(nome_csv) as stream:
                ler_chunks(stream)
    else:
        ler_chunks(io.BytesIO(conteudo))
    categoria_df = pd.concat(categorias, ignore_index=True) if categorias else pd.DataFrame(
        columns=["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)", "Ano"]
    )
    return totais, categoria_df


## 5. Identificar a situação eleitoral

In [ ]:
def identificar_eleitos(recurso, ids_candidatos):
    """Identifica candidatos com situação final contendo ELEITO processando todos os CSVs necessários de votação."""
    conteudo = baixar_conteudo(recurso)
    eleitos = set()

    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as arquivo_zip:
            # Procuramos primeiro o arquivo consolidado Brasil para poupar processamento
            nomes_csv = [nome for nome in arquivo_zip.namelist() if nome.lower().endswith((".csv", ".txt"))]

            # Se houver um arquivo unificado para o BRASIL, usamos ele
            brasil_csv = [n for n in nomes_csv if "brasil" in n.lower() or "_br" in n.lower()]
            arquivos_para_processar = brasil_csv if brasil_csv else nomes_csv

            print(f"Processando {len(arquivos_para_processar)} arquivo(s) de resultado no ZIP...")
            for nome_csv in arquivos_para_processar:
                # Apenas processa se for arquivo de votação nominal de candidato por município/zona ou similar
                if "votacao_candidato" in nome_csv.lower() or "votacao_nominal" in nome_csv.lower() or brasil_csv:
                    conteudo_csv = arquivo_zip.read(nome_csv)

                    for parte in pd.read_csv(
                        io.BytesIO(conteudo_csv),
                        sep=";",
                        encoding="latin1",
                        chunksize=250_000,
                        low_memory=False
                    ):
                        id_col = encontrar_coluna(parte, ["SQ_CANDIDATO"])
                        situacao_col = encontrar_coluna(parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"])

                        parte[id_col] = parte[id_col].astype(str).str.strip()
                        parte = parte[parte[id_col].isin(ids_candidatos)]

                        situacao = parte[situacao_col].fillna("").astype(str).str.upper()
                        filtro_eleito = (
                            situacao.str.contains("ELEITO")
                            & ~situacao.str.contains("NAO ELEITO")
                            & ~situacao.str.contains("NÃO ELEITO")
                        )
                        eleitos.update(parte.loc[filtro_eleito, id_col].unique())
    else:
        for parte in pd.read_csv(
            io.BytesIO(conteudo),
            sep=";",
            encoding="latin1",
            chunksize=250_000,
            low_memory=False
        ):
            id_col = encontrar_coluna(parte, ["SQ_CANDIDATO"])
            situacao_col = encontrar_coluna(parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"])

            parte[id_col] = parte[id_col].astype(str).str.strip()
            parte = parte[parte[id_col].isin(ids_candidatos)]

            situacao = parte[situacao_col].fillna("").astype(str).str.upper()
            filtro_eleito = (
                situacao.str.contains("ELEITO")
                & ~situacao.str.contains("NAO ELEITO")
                & ~situacao.str.contains("NÃO ELEITO")
            )
            eleitos.update(parte.loc[filtro_eleito, id_col].unique())

    return eleitos

## 6. Processar eleições de 2018, 2022 e 2026

In [ ]:
registros = []
erros = []
categorias_por_ano = {}

# Garantia: define a função nesta própria célula antes de processar qualquer ano.
def encontrar_col(df, alternativas):
    for nome in alternativas:
        if nome in df.columns:
            return nome
    raise ValueError(f"Colunas {alternativas} não encontradas. Colunas disponíveis: {list(df.columns)}")

for ano in ANOS:
    print("\n" + "=" * 65)
    print("PROCESSANDO:", ano)
    try:
        recursos = localizar_recursos(ano)
        candidatos = baixar_e_ler(recursos["candidatos"], tipo="candidatos")
        id_col = encontrar_coluna(candidatos, ["SQ_CANDIDATO"])
        nome_col = encontrar_coluna(candidatos, ["NM_URNA_CANDIDATO", "NM_CANDIDATO"])
        partido_col = encontrar_coluna(candidatos, ["SG_PARTIDO"])
        cargo_col = encontrar_coluna(candidatos, ["DS_CARGO"])
        uf_col = encontrar_coluna(candidatos, ["SG_UF"])
        candidatos[id_col] = candidatos[id_col].astype(str).str.strip()

        presidentes = candidatos[candidatos[cargo_col].astype(str).str.upper() == "PRESIDENTE"].copy()
        governadores = candidatos[
            (candidatos[cargo_col].astype(str).str.upper() == "GOVERNADOR")
            & candidatos[uf_col].isin(ESTADOS_NE)
        ].copy()
        selecionados = pd.concat([presidentes, governadores], ignore_index=True).drop_duplicates(subset=[id_col])
        ids = set(selecionados[id_col].dropna().astype(str))
        if not ids:
            raise ValueError("Nenhum presidente ou governador do Nordeste encontrado.")

        despesas, categorias = processar_despesas_candidatos(recursos["despesas"], ids, ano)
        categorias_por_ano[ano] = categorias
        eleitos = identificar_eleitos(recursos["resultados"], ids)
        for _, cand in selecionados.iterrows():
            cid = str(cand[id_col])
            registros.append({
                "Data da eleição": DATAS[ano], "Ano": ano, "Cargo": cand[cargo_col],
                "Estado": cand[uf_col], "SQ_CANDIDATO": cid, "Candidato": cand[nome_col],
                "Partido": cand[partido_col], "Gasto total (R$)": despesas.get(cid, 0),
                "Eleito": "Sim" if cid in eleitos else "Não",
            })
        print(f"Candidatos selecionados: {len(selecionados)}")
        print(f"Candidatos com despesas: {sum(despesas.get(cid, 0) > 0 for cid in ids)}")
        print("Processamento concluído.")
    except Exception as erro:
        erros.append({"Ano": ano, "Erro": f"{type(erro).__name__}: {erro}"})
        print("ERRO:", erros[-1]["Erro"])

print("\nResumo de erros:")
display(pd.DataFrame(erros))


## 7. Tabela geral de candidatos

In [ ]:
if erros:
    print("ATENÇÃO — anos com falha; os resultados abaixo são parciais:")
    display(pd.DataFrame(erros))

colunas_resultado = [
    "Data da eleição", "Ano", "Cargo", "Estado", "SQ_CANDIDATO",
    "Candidato", "Partido", "Gasto total (R$)", "Eleito"
]
tabela_geral = pd.DataFrame(registros, columns=colunas_resultado)
if tabela_geral.empty:
    print("Nenhum ano foi processado com sucesso. Não há dados para exibir.")
else:
    tabela_geral["Gasto total (R$)"] = pd.to_numeric(
        tabela_geral["Gasto total (R$)"], errors="coerce"
    ).fillna(0)
    tabela_geral = tabela_geral.drop_duplicates(
        subset=["Ano", "SQ_CANDIDATO"], keep="first"
    )
    tabela_geral = tabela_geral.sort_values(
        ["Ano", "Cargo", "Gasto total (R$)"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
    display(tabela_geral.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))


## 8. Os 10 candidatos que mais gastaram por ano e cargo


In [ ]:
if tabela_geral.empty:
    top10 = pd.DataFrame(columns=tabela_geral.columns)
    print("Top 10 indisponível: nenhum candidato foi processado.")
else:
    top10 = (
        tabela_geral.sort_values("Gasto total (R$)", ascending=False)
        .groupby(["Ano", "Cargo"], group_keys=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top10.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))


## 9. Onde foram gastos os recursos dos 10 maiores gastadores


In [ ]:
# Tabela detalhada das categorias de despesas contratadas para os Top 10.
if top10.empty:
    gastos_por_categoria = pd.DataFrame(columns=["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"])
    print("Top 10 vazio; tabela de categorias não pode ser calculada.")
else:
    blocos = []
    for ano in sorted(top10["Ano"].unique()):
        cat = categorias_por_ano.get(int(ano), pd.DataFrame())
        if cat.empty:
            continue
        ids_top = set(top10.loc[top10["Ano"] == ano, "SQ_CANDIDATO"].astype(str))
        cat = cat[cat["SQ_CANDIDATO"].astype(str).isin(ids_top)].copy()
        nomes = top10.loc[top10["Ano"] == ano, ["SQ_CANDIDATO", "Candidato"]].drop_duplicates()
        cat = cat.merge(nomes, on="SQ_CANDIDATO", how="left")
        blocos.append(cat[["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"]])
    if blocos:
        gastos_por_categoria = pd.concat(blocos, ignore_index=True).groupby(
            ["Ano", "Candidato", "Categoria de despesa"], as_index=False
        )["Valor gasto (R$)"].sum().sort_values(
            ["Ano", "Candidato", "Valor gasto (R$)"], ascending=[True, True, False]
        )
        display(gastos_por_categoria.style.format({"Valor gasto (R$)": "R$ {:,.2f}"}))
    else:
        gastos_por_categoria = pd.DataFrame(columns=["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"])
        print("Nenhuma despesa de candidato selecionado foi encontrada.")


## 10. Exportar as tabelas para Excel

In [ ]:
arquivo_excel = "analise_gastos_eleitorais.xlsx"

with pd.ExcelWriter(arquivo_excel) as writer:
    tabela_geral.to_excel(
        writer, sheet_name="Todos os candidatos", index=False
    )
    top10.to_excel(
        writer, sheet_name="Top 10 gastos", index=False
    )
    gastos_por_categoria.to_excel(
        writer, sheet_name="Despesas por categoria", index=False
    )

print("Planilha criada:", arquivo_excel)

# Se estiver executando no Google Colab, inicia o download.
try:
    from google.colab import files
    files.download(arquivo_excel)
except ImportError:
    print("Ambiente fora do Colab. O arquivo ficou salvo na pasta atual.")


## 11. Cuidados antes de apresentar os resultados

- Confira se todos os três anos foram processados sem erros.
- Confira se o CSV selecionado para despesas é realmente o de despesas de candidatos.
- Confira se o arquivo de resultados contém a situação final do turno, e não apenas resultados parciais.
- Em eleições com segundo turno, a situação de eleito deve refletir o resultado final.
- A tabela Top 10 é separada por ano e cargo; assim, não mistura candidatos a presidente com candidatos a governador.
- A seção 9 detalha as categorias de despesas dos 10 maiores gastadores.
- Despesa declarada não é a mesma coisa que dinheiro arrecadado. Aqui, o indicador é a soma de `VR_DESPESA_CONTRATADA`, caso essa coluna exista no arquivo selecionado.
